In [1]:
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import statsmodels.formula.api as smf

import sys; sys.path.insert(0, '..')
from src.palette import register, content_colors, colorway, CONTENT_ORDER

CONTENT_COLORS = register('light')

In [2]:
conn = sqlite3.connect('../data/lafc_content.db')

In [3]:
with open('../sql/videos_vs_lafc_match_context.sql') as f:
    query = f.read()

df = pd.read_sql(query, conn)
df = df[df['published_at'] >= '2025-01-01T00:00:00Z'].copy()
df['content_type'] = df['content_type'].fillna('no_playlist')
df['playlist'] = df['playlist'].fillna('(no playlist)')
df

,video_id,title,description,published_at,duration,view_count,like_count,comment_count,engagement_rate,format,...,goals_for,goals_against,lafc_points,lafc_played,lafc_wins,opp_points,opp_played,opp_wins,days_since_match,days_until_match
0,IxrFLgowFd4,Armindo Sieb is Black & Gold.,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T16:40:02Z,PT52S,275,23,7,0.10909,horizontal,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.72,NaN
1,HugEGKBw0kk,LAFC vs QRO | Postmatch Media,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T09:02:51Z,PT14M19S,306,22,26,0.15686,horizontal,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.40,NaN
2,pLVoxNTyGLI,The top scorer in Leagues Cup history 📈,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:30:24Z,PT15S,6287,169,12,0.02879,short,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.33,NaN
3,wz6UrdGQWjY,BOUANGA EQUALIZER 💥,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:15:01Z,PT13S,3940,91,4,0.02411,short,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.32,NaN
4,SqgJkPzCN6Y,Denis Bouanga equalizes against Querétaro,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:02:06Z,PT13S,1712,55,6,0.03563,horizontal,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.31,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210,zPYzE9H0Zyo,Igor Jesus is Black & Gold,📝 #LAFC acquires Igor Jesus from Portuguese Pr...,2025-01-21T22:36:06Z,PT1M24S,1221,43,5,0.03931,horizontal,...,3.0,1.0,61.0,33.0,18.0,21.0,33.0,6.0,93.90,31.96
1211,Uycrl281Zjg,"Part of our History | Thank you, Erik Dueñas",The LAFC Original. Forever Black & Gold.\n\nBe...,2025-01-14T19:59:43Z,PT1M30S,1518,43,6,0.03228,horizontal,...,3.0,1.0,61.0,33.0,18.0,21.0,33.0,6.0,86.79,39.07
1212,xkQPq_y1QOg,And now for some midfield thunder 🔨⚡,Odin Thiago Holm is Balck & Gold.\n\nWatch LAF...,2025-01-13T19:44:09Z,PT19S,2473,143,12,0.06268,short,...,3.0,1.0,61.0,33.0,18.0,21.0,33.0,6.0,85.78,40.08
1213,bm7P0aOOhAo,Odin Thiago Holm is Black & Gold,LAFC has acquired Norwegian midfielder Odin Th...,2025-01-13T19:34:11Z,PT1M23S,2684,67,8,0.02794,horizontal,...,3.0,1.0,61.0,33.0,18.0,21.0,33.0,6.0,85.77,40.09


## Distributions

View counts span roughly 100 to over a million, so they are shown on a log scale
after the first histogram.
Engagement rate doesn't need a log transform as its span is much smaller.

In [ ]:

print(df['view_count'].mean())
print('View Count Median=' df['view_count'].median())


SyntaxError: invalid syntax. Perhaps you forgot a comma? (1245459785.py, line 2)

In [6]:
fig = px.histogram(
    df, x='view_count',
    title="Videos per view - linear scale"
    )

fig.update_xaxes(title='Views')

fig.update_yaxes(title='Number of videos')

fig.show()

In [7]:
df['log10_views'] = np.log10(df['view_count'])

fig = px.histogram(
    df, x='log10_views',
    nbins=80,
    title='Videos per View - Log Scale', 
    )
fig.update_xaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'],
    title='Views')

fig.update_yaxes(
    title='Number of videos'
)

fig.show()

In [ ]:
fig = px.histogram(
    df, x='engagement_rate',
    nbins=80,
    title= 'Engagement rate per video'
    )

fig.update_yaxes()

fig.update_xaxes(tickformat='.1%')

fig.show()

**The two metrics are shaped completely differently.** Views span ~100 to 1.1M,
median 2,843 against a mean of 28,065 - a handful of viral videos dominate the
average, which is why every cut below uses the median and every view chart is on
a log scale. Engagement rate spans 0.5% to 18%, median 5.13%: mild enough to
read raw, so it is never logged.

This asymmetry is the first hint of what Finding 10 turns into: reach is
lottery-shaped, reaction is not.

## Format and content type

Format is a *delivery surface*, not a content category - the channel tabs say
where YouTube puts a video, not what kind of video it is. `live` and `horizontal`
are both long-form 16x9 video and are collapsed into one group; `short` is the
genuinely different surface.

Content type is then examined *within* each format, because 86% of Shorts sit in
no playlist at all - so a pooled comparison would mostly be measuring format.

In [9]:
df['format'].value_counts()

format
horizontal    741
short         398
live           76
Name: count, dtype: int64

In [10]:
df.groupby('format')['view_count'].median().sort_values(ascending=False)

format
short         8331.0
live          2023.0
horizontal    1700.0
Name: view_count, dtype: float64

In [11]:
df['playback_type'] = np.where(df['format'] == 'short', 'short', 'horizontal+live')

order = ['short', 'horizontal+live']
n = df['playback_type'].value_counts()

fig = px.box(
    df, x='playback_type', y='view_count',
    log_y=True,
    color='playback_type',
    category_orders={'playback_type': order},
    title='View Count by Playback Type - Short vs. Horizontal+Live',
    labels={'playback_type': '', 'view_count': 'View Count'},
)

fig.update_traces(marker=dict(opacity=0.3, size=4), jitter=0.4)

fig.update_layout(showlegend=False)

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(
    tickvals=[100, 1_000, 10_000, 100_000, 1_000_000],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()

In [12]:
order = ['short', 'horizontal+live']
n = df['playback_type'].value_counts()

fig = px.box(
    df, x='playback_type', y='engagement_rate',
    color='playback_type',
    category_orders={'playback_type': order},
    title='Engagement Rate by Playback Type - Short vs. Horizontal+Live',
    labels={'playback_type': '', 'engagement_rate': 'Engagement Rate'},
)

fig.update_traces(marker=dict(opacity=0.3, size=4), jitter=0.4)

fig.update_layout(showlegend=False)

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(tickformat='.1%')

fig.show()

In [13]:
table = pd.crosstab(df['format'], df['content_type'])
table

content_type,community,feature,full_match,highlights,match_preview,no_playlist,podcast,press_interview,show,unclassified
format,,,,,,,,,,
horizontal,1,5,6,171,30,57,184,155,82,50
live,0,0,2,0,0,3,69,0,0,2
short,0,0,0,46,1,322,0,0,0,29


In [14]:
shorts_df = df[df['format'] == 'short'].copy()
playlist_table = shorts_df['playlist'].value_counts(dropna=False)
content_type_table = shorts_df['content_type'].value_counts(dropna=False)

display(content_type_table)
display(playlist_table)

content_type
no_playlist      322
highlights        46
unclassified      29
match_preview      1
Name: count, dtype: int64

playlist
(no playlist)        322
Highlights            46
The Son Spotlight     29
Match Previews         1
Name: count, dtype: int64

In [15]:
order = shorts_df.groupby('content_type')['view_count'].median().sort_values().index.tolist()
n = shorts_df['content_type'].value_counts()

fig = px.box(
    shorts_df, x='content_type', y='log10_views',
    color='content_type',
    category_orders={'content_type': order},
    title='Views (log 10) by Content Type - Short Format',
    labels={'content_type': 'Content Type', 'log10_views': "Views"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()

In [16]:
order = shorts_df.groupby('content_type')['engagement_rate'].median().sort_values().index.tolist()
n = shorts_df['content_type'].value_counts()

fig = px.box(
    shorts_df, x='content_type', y='engagement_rate',
    color='content_type',
    category_orders={'content_type': order},
    title='Engagement Rate (log 10) by Content Type - Short Format',
    labels={'content_type': 'Content Type', 'engagment_rate': "Engagement Rate"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(tickformat='.1%')

fig.show()

In [17]:
unclassified_shorts_df = shorts_df[shorts_df['content_type'] == 'unclassified'].copy()
unclassified_shorts_df['playlist'].value_counts(dropna=False)

playlist
The Son Spotlight    29
Name: count, dtype: int64

In [18]:
horizontal_df = df[df['format'] != 'short'].copy()

order = horizontal_df.groupby('content_type')['view_count'].median().sort_values().index.tolist()
n = horizontal_df['content_type'].value_counts()

fig = px.box(
    horizontal_df, x='content_type', y='log10_views',
    color='content_type',
    color_discrete_map=CONTENT_COLORS,
    category_orders={'content_type': order},
    title='Views (log 10) by Content Type - Horizontal / Live ',
    labels={'content_type': 'Content Type', 'log10_views': "Views"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()

In [19]:
order = horizontal_df.groupby('content_type')['engagement_rate'].median().sort_values().index.tolist()
n = horizontal_df['content_type'].value_counts()

fig = px.box(
    horizontal_df, x='content_type', y='engagement_rate',
    color='content_type',
    color_discrete_map=CONTENT_COLORS,
    category_orders={'content_type': order},
    title='Engagement Rate by Content Type - Horizontal / Live',
    labels={'content_type': 'Content Type', 'engagement_rate': "Engagement Rate"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(tickformat='.1%')

fig.show()

In [20]:
unclassified_horizontal_df = horizontal_df[horizontal_df['content_type'] == 'unclassified'].copy()
unclassified_horizontal_df['playlist'].value_counts(dropna=False)

playlist
The Son Spotlight    40
Major News            8
The Vela Vault        4
Name: count, dtype: int64

**Format is the biggest single divide in the data.** Shorts get ×4.7 the
median views of horizontal+live (8,331 vs 1,757) and 1.2 points *less*
engagement (4.30% vs 5.52%) - the trade-off that recurs at every level below.

**Content type had to be examined within format**, because the two are nearly
collinear: three content types have zero Shorts, and 86% of Shorts sit in no
playlist at all. Pooling them would mean measuring format and calling it content.

The one group that would not behave was `unclassified` - which turned out to be
almost entirely one playlist.

## The Son Spotlight

The single largest effect in the data. This section establishes the size, then
tests the two obvious objections: that it is really a Shorts effect, and that it
is really a "player content" effect rather than Son specifically.

In [ ]:
df['is_son'] = df['playlist'] == 'The Son Spotlight'

df.groupby('is_son').agg(
    n=('view_count', 'size'),
    median_views=('view_count', 'median'),
    median_engagement=('engagement_rate', 'median'))

In [ ]:
# Son Spotlight is Shorts-heavy, so the gap could be a format effect.
# Comparing within each format removes that explanation.
df.groupby(['playback_type', 'is_son']).agg(
    n=('view_count', 'size'),
    median_views=('view_count', 'median'))

In [ ]:
# Is it Son, or player content generally? The Vela Vault is the only
# comparable player-subject playlist - and it has very few videos in this window.
df[df['playlist'].isin(['The Son Spotlight', 'The Vela Vault'])].groupby('playlist').agg(
    n=('view_count', 'size'),
    median_views=('view_count', 'median'))

In [21]:
son_spotlight_df = unclassified_horizontal_df[unclassified_horizontal_df['playlist'] == 'The Son Spotlight'].copy()
son_spotlight_df[['title', 'playlist']]

,title,playlist
20,One Year of Sonny Goals,The Son Spotlight
36,Sonny vs. SKC | EVERY ANGLE,The Son Spotlight
52,SONNY SCORES HIS 3RD GOAL IN 3 MATCHES | LAFC ...,The Son Spotlight
54,Sonny vs RSL | EVERY ANGLE,The Son Spotlight
67,SONNY SCORES FROM THE TOP OF THE BOX | LAFC vs...,The Son Spotlight
79,Son Heung-Min | EVERY ANGLE of his derby goal ...,The Son Spotlight
92,SONNY’S FIRST GOAL OF THE SEASON | LAG vs LAFC,The Son Spotlight
112,Who has the most aura? w/ Sonny | Ryan & Aaron...,The Son Spotlight
120,Sonny bobblehead arrives in Los Angeles | LAFC...,The Son Spotlight
236,Sonny's goal from the stands | LAFC vs Cruz Azul,The Son Spotlight


**Finding.** Son Spotlight videos get ~32x the median views of everything else,
and it holds in both formats - x38 long-form, x24 Shorts - so it is not a Shorts
artifact. In the regression below it is x19 after controlling for format,
content type and timing, with no engagement penalty (+1.39 pts, p=0.156).

**What this cannot tell us.** Whether this is player content generally or Son
specifically. The Vela Vault is the only control and has 4 videos in this window.

In [22]:
order = horizontal_df.groupby('playlist')['view_count'].median().sort_values().index.tolist()
n = horizontal_df['playlist'].value_counts()

fig = px.box(
    horizontal_df, y='playlist', x='view_count',
    log_x=True,
    color='playlist',
    category_orders={'playlist': order},
    title='View Count (Log Scale) by Playlist - Horizontal / Live',
    labels={'playlist': 'Playlist', 'view_count': 'View Count'},
    height=600,
)

fig.update_layout(showlegend=False)

fig.update_yaxes(
    tickvals=order,
    ticktext=[f'{p}  (n={n[p]})' for p in order])

fig.update_xaxes(
    tickvals=[100, 1_000, 10_000, 100_000, 1_000_000],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()


In [33]:
order = horizontal_df.groupby('playlist')['engagement_rate'].median().sort_values().index.tolist()
n = horizontal_df['playlist'].value_counts()

fig = px.box(
    horizontal_df, y='playlist', x='engagement_rate',
    color='playlist',
    category_orders={'playlist': order},
    title='Engagement Rate by Playlist - Horizontal / Live',
    labels={'playlist': 'Playlist', 'engagement_rate': 'Engagement Rate'},
    height=600,
)

fig.update_layout(showlegend=False)

fig.update_yaxes(
    tickvals=order,
    ticktext=[f'{p}  (n={n[p]})' for p in order])

fig.update_xaxes(tickformat='.1%')

fig.show()


## Timing

In [24]:
# Assigns the two columns into two series.
after  = df['days_since_match']    # days SINCE the previous match (always ≥ 0)
before = df['days_until_match']    # days UNTIL the next match     (always ≥ 0)


# Fills in na with infinity, then compares the two series, row by row, and returns a boolean. If before is smaller - then it returns true, meaning this row is closer to the NEXT match.
closer_to_next = before.fillna(np.inf) < after.fillna(np.inf)

# Assigns either a -before or after based on the boolean in closer_to_next
df['days_from_match'] = np.where(closer_to_next, -before, after)

#Overwrite nan if its been 21 days since (and until) the closest matches

OFFSEASON_DAYS = 21
not_in_cycle = ((after.fillna(np.inf)  > OFFSEASON_DAYS) &
                (before.fillna(np.inf) > OFFSEASON_DAYS))

df.loc[not_in_cycle, 'days_from_match'] = np.nan

print('not in a cycle :', not_in_cycle.sum())
print('before a match :', (df['days_from_match'] < 0).sum())
print('after a match  :', (df['days_from_match'] > 0).sum())

not in a cycle : 71
before a match : 485
after a match  : 659


In [25]:
#Binning and adding the bin info back to the df.

CYCLE_EDGES  = [-np.inf, -4, -3, -2, -1, 0, 1, 2, 3, 4, np.inf]
CYCLE_LABELS = ['4+ before', '3-4 before', '2-3 before', '1-2 before', '0-1 before',
                '0-1 after', '1-2 after', '2-3 after', '3-4 after', '4+ after']

df['cycle_bin'] = pd.cut(
    df['days_from_match'],
    bins=CYCLE_EDGES,
    labels=CYCLE_LABELS, right=False
    )

display(df[['title', 'days_since_match', 'days_until_match', 'cycle_bin']].sort_values('days_since_match'))

,title,days_since_match,days_until_match,cycle_bin
52,SONNY SCORES HIS 3RD GOAL IN 3 MATCHES | LAFC ...,0.02,6.85,0-1 after
811,Black & Gold Insider Ep. 21 | Christian Beceri...,0.02,2.98,0-1 after
67,SONNY SCORES FROM THE TOP OF THE BOX | LAFC vs...,0.02,2.98,0-1 after
575,SONNY SCORES IN THE OPENING MINUTES,0.03,3.95,0-1 after
685,HollingsheadER 👨🏻,0.03,6.93,0-1 after
...,...,...,...,...
1160,LAFC vs MIN | Pre Match Media,125.04,0.82,0-1 before
1159,The guys reacting to Denis Bouanga's bobblehead,125.10,0.76,0-1 before
1158,Starboy Season Loading ⏳,125.16,0.70,0-1 before
347,The season starts today 🏟️,125.73,0.33,0-1 before


In [26]:
print(df['cycle_bin'].value_counts(dropna=False).sort_index())

cycle_bin
4+ before      85
3-4 before     62
2-3 before     90
1-2 before    133
0-1 before    115
0-1 after     341
1-2 after      82
2-3 after      68
3-4 after      49
4+ after      119
NaN            71
Name: count, dtype: int64


In [27]:
PLOT_BINS = CYCLE_LABELS[1:-1]      #Drops first and last bins, since they are farthest away from kickoffs

KICKOFF_X = PLOT_BINS.index('0-1 after') - 0.5   #Sets kickoff line on charts below

# Drops off-season videos and videos with no engagment rate (although none exist, but just in case)

plot_df = df[df['cycle_bin'].isin(PLOT_BINS)].dropna(subset=['engagement_rate']).copy()
plot_df['cycle_bin'] = plot_df['cycle_bin'].cat.remove_unused_categories()
n = plot_df['cycle_bin'].value_counts()

In [28]:
fig = px.box(
    plot_df, x='cycle_bin', y='view_count',
    log_y=True,
    category_orders={'cycle_bin': PLOT_BINS},
    color='cycle_bin',
    points=False,
    title='View Count Across the Match Cycle',
    labels={'cycle_bin': 'Position in match cycle',
            'view_count': 'View Count (log)'},
)

fig.update_xaxes(
    tickvals=PLOT_BINS,
    ticktext=[f'{b}<br>n={n[b]}' for b in PLOT_BINS])

fig.add_vline(x=KICKOFF_X, line_dash='dot', line_color='#888888',
              annotation_text='kickoff', annotation_position='top')

fig.show()

In [29]:
fig = px.box(
    plot_df, x='cycle_bin', y='engagement_rate',
    color='cycle_bin',
    category_orders={'cycle_bin': PLOT_BINS},
    points=False,
    title='Engagement Rate Across the Match Cycle',
    labels={'cycle_bin': 'Position in match cycle',
            'engagement_rate': 'Engagement Rate'},
)

fig.update_xaxes(
    tickvals=PLOT_BINS,
    ticktext=[f'{b}<br>n={n[b]}' for b in PLOT_BINS])

fig.update_yaxes(tickformat='.1%')

fig.add_vline(x=KICKOFF_X, line_dash='dot', line_color='#888888',
              annotation_text='kickoff', annotation_position='top')

fig.show()

In [30]:
table = plot_df.groupby('cycle_bin', observed=True).agg(
        n=('view_count', 'size'),
        median_views=('view_count', 'median'),
        median_engagement_rate=('engagement_rate', 'median'))

table['median_engagement_rate'] = (table['median_engagement_rate'] * 100).round(2).astype(str) + '%'

table

,n,median_views,median_engagement_rate
cycle_bin,,,
3-4 before,62,2444.0,4.78%
2-3 before,90,2032.0,6.02%
1-2 before,133,2301.0,5.69%
0-1 before,115,2180.0,5.98%
0-1 after,341,4979.0,3.94%
1-2 after,82,2808.0,5.45%
2-3 after,68,1768.0,5.38%
3-4 after,49,2082.0,5.69%


**One bin does all the work.** Everything before kickoff is flat - 2,032 to
2,444 median views across four days, no trend. Then the 24 hours after kickoff
jumps to 4,979 and decays to 2,808 within a day, 1,768 by day two.

**Engagement runs exactly opposite.** Its peak is the final 24 hours *before*
kickoff (5.98%) and its trough is the 24 hours after (3.94%). Anticipation
drives comments; the result drives views.

Two notes on method. The bins are symmetric one-day windows because the earlier
asymmetric version had no 1-day bin after kickoff, which averaged the spike with
its own decay and hid it. And the open-ended end bins are dropped: before/after
is assigned by whichever match is *nearer*, so a video can only land beyond ±4
days when there was a long schedule gap - making those bins break-period content
rather than a position in the cycle.

## Regression

### OLS: log₁₀ views on format, content type, and match-cycle position
Baseline: a long-form highlights video published 0–1 days before a match — 1,932 predicted views.

In [31]:
df['is_short'] = (df['format'] == 'short')
df['is_son']   = (df['playlist'] == 'The Son Spotlight')

# Dropped on sample size (community n=1, full_match n=8, feature n=5)
TOO_SMALL = ['community', 'full_match', 'feature']
model_df = df[~df['content_type'].isin(TOO_SMALL)].copy()

# Baseline: a long-form highlights video published 0-1 days BEFORE a match.
# Every coefficient is relative to that.
#
# These two references were chosen to make the output readable:
#
#   highlights   large (n=217) so the baseline is precisely measured, and the
#                best-performing real type - so every coefficient reads as
#                "how far below the best format". Not no_playlist, despite
#                being larger, because "vs videos we couldn't label" is a
#                meaningless anchor.
#   0-1 before   the floor of the cycle, so all nine coefficients come out
#                positive and read as lift over the worst window.
# References are also named because C() otherwise picks the alphabetically
# first level - which was community, n=1, making every content_type
# coefficient a comparison against a single video.
model = smf.ols(
    'log10_views ~ is_short + is_son'
    ' + C(content_type, Treatment(reference="highlights"))'
    ' + C(cycle_bin, Treatment(reference="0-1 before"))',
    data=model_df).fit()

# statsmodels silently drops rows with a NaN in any model column - here the
# offseason videos, where days_from_match is NaN.
print(f'rows {len(df)} -> {len(model_df)} after dropping tiny types '
      f'-> {int(model.nobs)} used in the model')

print(model.summary())

rows 1215 -> 1201 after dropping tiny types -> 1131 used in the model
                            OLS Regression Results                            
Dep. Variable:            log10_views   R-squared:                       0.445
Model:                            OLS   Adj. R-squared:                  0.437
Method:                 Least Squares   F-statistic:                     52.57
Date:                Fri, 21 Aug 2026   Prob (F-statistic):          2.48e-129
Time:                        14:18:36   Log-Likelihood:                -1013.2
No. Observations:                1131   AIC:                             2062.
Df Residuals:                    1113   BIC:                             2153.
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                                                                            coef    std err          t      P>|t|      [0.025      0.975]
--

**Three effects survive being tested against each other** (R² 0.445, n=1,131):
Shorts ×3.2, Son ×19.5, and the post-match window ×2.4 on day one and ×2.0 on
day two - gone by day three (p=0.29). So the window is **48 hours, not 24**.

Content type separates cleanly against `highlights`: podcast ×0.33, press
interview ×0.42, show ×0.49, all p<0.001. `match_preview` at ×1.09 (p=0.77) is
statistically identical to highlights - the *timing* of pre-match content hurts,
the format does not.

**R² of 0.445 is high for this question but the model is not predictive.**
Typical prediction error is ×3.9. It describes averages across many videos,
which is what a content strategy needs; it cannot forecast a single video.

### OLS: Engagement Rate on format, content type, and match-cycle position
Baseline: a long-form highlights video published 0–1 days before a match — 1,932 predicted views.

In [32]:
engagement_model = smf.ols(
    'engagement_rate ~ is_short + is_son'
    ' + C(content_type, Treatment(reference="highlights"))'
    ' + C(cycle_bin, Treatment(reference="0-1 before"))',
    data=model_df).fit()

print(engagement_model.summary())

                            OLS Regression Results                            
Dep. Variable:        engagement_rate   R-squared:                       0.191
Model:                            OLS   Adj. R-squared:                  0.179
Method:                 Least Squares   F-statistic:                     15.47
Date:                Fri, 21 Aug 2026   Prob (F-statistic):           6.86e-41
Time:                        14:18:36   Log-Likelihood:                 2605.7
No. Observations:                1131   AIC:                            -5175.
Df Residuals:                    1113   BIC:                            -5085.
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                                                                            coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------

**Almost every sign flips.** Shorts get ×3.2 the views and **1.3 points less**
engagement. Every content type loses to `highlights` on views and beats it on
engagement - podcast most of all at +2.6 points while getting a third of the
views. The 24 hours after kickoff is the views peak and the engagement trough.

**`is_son` is the exception**: +1.39 points at p=0.156 - not significant, so "no
engagement penalty" rather than "a gain". It is the only lever in either model
without a trade-off.

**R² drops from 0.445 to 0.191.** These predictors say much more about who
*sees* a video than about who *reacts* to it - most of what drives engagement is
not in this dataset.

## Duration, within long-form

Finding 3 said short videos win, but flagged its own confound: duration is a
proxy for content *type*, so "short wins" might just mean "clips beat shows".
Shorts are excluded here - they are all under a minute by definition, so
including them would rediscover the format effect rather than test duration.

In [ ]:
long_df = df[df['format'] != 'short'].copy()
long_df['dur_min']   = pd.to_timedelta(long_df['duration']).dt.total_seconds() / 60
long_df['log10_dur'] = np.log10(long_df['dur_min'].clip(lower=0.1))

BANDS  = [0, 1, 3, 10, 30, 60, np.inf]
LABELS = ['<1m', '1-3m', '3-10m', '10-30m', '30-60m', '60m+']
long_df['dur_band'] = pd.cut(long_df['dur_min'], bins=BANDS, labels=LABELS, right=False)

long_df.groupby('dur_band', observed=True).agg(
    n=('view_count', 'size'),
    median_views=('view_count', 'median'),
    median_engagement=('engagement_rate', 'median'))

In [ ]:
# Duration is nearly collinear with content type in long-form - highlights run
# ~1 minute, podcasts ~36. So the raw relationship may be measuring type, not length.
long_df.groupby('content_type')['dur_min'].agg(['size', 'median']).sort_values('median')

In [ ]:
# The same coefficient with and without a content_type control.
lf = long_df[~long_df['content_type'].isin(TOO_SMALL)]
CT = 'C(content_type, Treatment(reference=\"highlights\"))'

for dv in ['log10_views', 'engagement_rate']:
    alone = smf.ols(f'{dv} ~ log10_dur + is_son', data=lf).fit()
    ctrl  = smf.ols(f'{dv} ~ log10_dur + is_son + {CT}', data=lf).fit()
    print(f'{dv:16} alone {alone.params["log10_dur"]:+.4f} (p={alone.pvalues["log10_dur"]:.4f})'
          f'   with content_type {ctrl.params["log10_dur"]:+.4f} (p={ctrl.pvalues["log10_dur"]:.4f})')

In [ ]:
# A sign flip on control is the signature of collinearity, so check it INSIDE
# single content types, where there is no type variation left to confound it.
for ct in ['highlights', 'press_interview', 'podcast', 'show']:
    sub = long_df[long_df['content_type'] == ct]
    v = smf.ols('log10_views ~ log10_dur + is_son', data=sub).fit()
    e = smf.ols('engagement_rate ~ log10_dur + is_son', data=sub).fit()
    print(f'{ct:16} n={len(sub):3}  IQR {sub.dur_min.quantile(.25):5.1f}-{sub.dur_min.quantile(.75):5.1f}m'
          f'   views {v.params["log10_dur"]:+.3f} p={v.pvalues["log10_dur"]:.3f}'
          f'   eng {e.params["log10_dur"]*100:+.2f}pts p={e.pvalues["log10_dur"]:.3f}')

**Duration and content type are nearly the same variable in long-form.**
Highlights run ~1 minute, podcasts ~36. Raw, longer means fewer views
(-0.19 per log minute) and more engagement - but both coefficients **flip sign**
once content type is controlled, which is the classic signature of collinearity
rather than a real reversal.

**Checking inside single content types, where no type variation is left:**

| content type | n | duration IQR | views per log10 minute |
|---|---|---|---|
| `highlights` | 171 | 0.9-5.8m | **+0.78** (p<0.001) |
| `press_interview` | 155 | 12.0-23.4m | **+0.55** (p<0.001) |
| `podcast` | 253 | 22.4-45.1m | +0.21 (n.s.) |
| `show` | 82 | 5.1-21.0m | **-0.50** (p=0.047) |

So it is real but **not universal**. A highlights package that is 10x longer
gets roughly 6x the views - a full match package beats a single goal clip.
The same holds for press interviews. It is null for podcasts and *reverses*
for shows.

**This resolves Finding 3's caveat.** "Short-form wins" is a statement about
the Shorts *surface*, not about length. Within long-form, longer is generally
better - the opposite of what a naive reading of Finding 3 would suggest.

Longer highlights also cost engagement (-1.61 pts per log minute, p<0.001),
which is the Finding 10 trade-off appearing at yet another level.

## Context: does the season matter?

Everything above is a lever the content team controls. These are the things they
do not - result, opponent quality, league position. All three come back null,
which is itself the finding: on-field context barely moves content performance.

In [ ]:
# Home vs away, on post-match videos where venue could plausibly matter.
post = df[df['days_from_match'].between(0, 2)]

post.groupby('home_away').agg(
    n=('view_count', 'size'),
    median_views=('view_count', 'median'))

In [ ]:
# Match result, with controls. The raw medians favour wins, but that gap is
# partly a content-mix effect - different videos get made after a loss.
result_model = smf.ols(
    'log10_views ~ C(result) + is_short + is_son'
    ' + C(content_type, Treatment(reference="highlights"))',
    data=post[~post['content_type'].isin(TOO_SMALL)]).fit()

print(result_model.summary().tables[1])

In [ ]:
# League position: LAFC form and opponent strength at the time of the nearest
# match. points-per-game rather than total points, since teams play unequal
# numbers of games (see docs/data_caveats.md).
standings = df[df['lafc_played'] > 0].copy()
standings['lafc_ppg'] = standings['lafc_points'] / standings['lafc_played']
standings['opp_ppg']  = standings['opp_points'] / standings['opp_played'].replace(0, np.nan)

standings_model = smf.ols(
    'log10_views ~ lafc_ppg + opp_ppg + is_short + is_son'
    ' + C(content_type, Treatment(reference="highlights"))',
    data=standings[~standings['content_type'].isin(TOO_SMALL)]).fit()

print(standings_model.summary().tables[1])

In [ ]:
# Opponent strength is significant pooled - but does it hold in the window
# where it could actually operate? If not, it is a scheduling artifact.
post_standings = standings[standings['days_from_match'].between(0, 2)]

smf.ols(
    'log10_views ~ lafc_ppg + opp_ppg + is_short + is_son'
    ' + C(content_type, Treatment(reference="highlights"))',
    data=post_standings[~post_standings['content_type'].isin(TOO_SMALL)]
).fit().summary().tables[1]

In [ ]:
# The away advantage looks real until you notice WHERE Son's content sits.
# Word boundary on 'son' matters - a loose match also catches 'Season'.
post = post.copy()
post['son_related'] = post['title'].str.contains(r'\bson\b|sonny|heung',
                                                 case=False, na=False)

post.groupby(['home_away', 'son_related']).agg(
    n=('view_count', 'size'),
    median_views=('view_count', 'median'))

**Context barely moves content performance - with one apparent exception that
dissolves on inspection.**

**Match result does not survive controls** - win x1.34, loss x0.71, neither
significant on 420 post-match videos. The raw gap is partly a content-mix
effect: different videos get made after a loss.

**LAFC form predicts nothing** (p=0.155). Opponent strength is significant in
the pooled model (p=0.007) but disappears in the post-match window where the
mechanism would have to operate (p=0.487) - so it is picking up something about
fixture scheduling, not opponent quality.

**Home vs away looked like a real effect and is not one.** Away videos appear to
get x1.9 the views. But 25% of away post-match videos are about Son against 7%
of home ones - his debut and early goals came away. Split it out and the raw
medians are near-identical (away 3,108, home 2,939). Year by year, 2025 is the
only season where away beats home; 2022, 2023 and 2026 all favour home. Not a
lever.

**One trap avoided.** `lafc_played` correlates with views at r=+0.311, but it is
a clock, not a standings effect - games 20-30 of 2025 is exactly when Son
arrived. Modelling it without `is_son` would report a strong "later in the
season is better" finding that is entirely one player.

## Synthesis

One chart for the finding the two models share. Built from **medians, not model
coefficients**, so it does not control for format or timing - the regressions
are what establish that the pattern holds under controls. This shows the shape.

In [36]:
# --- Synthesis: reach and engagement pull in opposite directions ---
summary = (df.groupby('content_type')
             .agg(n=('view_count', 'size'),
                  median_views=('view_count', 'median'),
                  median_eng=('engagement_rate', 'median'))
             .query('n >= 20')
             .reset_index())

fig = px.scatter(
    summary, x='median_views', y='median_eng',
    log_x=True,
    size='n', size_max=45,
    color='content_type',
    color_discrete_map=CONTENT_COLORS,
    text='content_type',
    title='Reach and engagement pull against each other',
    labels={'median_views': 'Median views (log)',
            'median_eng': 'Median engagement rate'},
    height=560,
)

fig.update_traces(textposition='top center')
fig.update_layout(showlegend=False)   # points are directly labelled

fig.update_xaxes(tickvals=[1_000, 10_000, 100_000],
                 ticktext=['1K', '10K', '100K'])
fig.update_yaxes(tickformat='.1%')

display(summary)
fig.show()


,content_type,n,median_views,median_eng
0,highlights,217,4014.0,0.032280
1,match_preview,31,2374.0,0.057690
2,no_playlist,382,5813.5,0.048370
3,podcast,253,1033.0,0.069270
4,press_interview,155,1218.0,0.051280
5,show,82,1148.5,0.052865
6,unclassified,81,62239.0,0.054860


## What this analysis found

**Reach and engagement are different goals, and most levers trade one for the
other.** Shorts get ×3.2 the views and 1.3 points less engagement. Every content
type loses to highlights on views and beats it on engagement. The 24 hours after
kickoff is the views peak and the engagement trough. Judging a podcast on views,
or a highlights clip on comment rate, misreads both.

**The post-match window is 48 hours, not 24.** ×2.4 on day one, ×2.0 on day two,
gone by day three. Everything before kickoff is flat - the final 24 hours before
a match is the weakest slot in the cycle for views, and the strongest for
engagement.

**"Short-form wins" is about the Shorts surface, not about length.** Within
long-form the relationship reverses: a highlights package 10× longer gets
roughly 6× the views. The lever is *publish to the Shorts feed*, not *make
everything shorter*.

**One player is the largest effect in the data** - ×19 views after controls,
and the only lever with no engagement penalty. Whether that is player content
generally or Son specifically cannot be answered here: the one comparable
playlist has 4 videos in this window.

**On-field context does not move content performance.** Result, league position
and opponent strength are all null. The apparent home/away effect dissolved once
Son's away-heavy content was separated out.

### What would change these conclusions

- **The Shorts probe.** 86% of Shorts have no playlist, so `content_type` is
  structurally a long-form label system. The segment that drives reach is the
  one with no labels.
- **A second star player.** Every subject finding rests on 69 videos and one
  arrival window.
- **More seasons.** This is 2025-01 to 2026-08. The 2024 era break shows how
  fast this channel's behaviour changes.
- **Multiple comparisons.** Seven or eight models were run against one dataset.
  The two main models are the trustworthy part; single results just under
  p=0.05 elsewhere should be treated as exploratory until re-tested.